# 02 — Model Selection: Theory to Decision Framework

## Part 1: Theory

### 1.1 Why Model Selection Matters

Choosing the wrong model costs enterprises **millions** in wasted compute, poor UX, or compliance failures.

### 1.2 Multi-Criteria Decision Analysis (MCDA)

No single metric determines the "best" model. We need **weighted multi-objective optimization**:

```
Score(model) = Σ(weight_i × normalized_metric_i)
```

Methods:
- **Weighted Sum Model (WSM)**: Simple linear combination
- **TOPSIS**: Distance from ideal/anti-ideal solutions
- **AHP**: Pairwise comparison of criteria importance

### 1.3 Pareto Optimality

A model is **Pareto-optimal** if no other model is better on ALL dimensions simultaneously.

```
Model A dominates Model B if:
  ∀i: metric_i(A) ≥ metric_i(B)  AND  ∃j: metric_j(A) > metric_j(B)
```

The **Pareto frontier** = set of all non-dominated models = your shortlist.

### 1.4 Total Cost of Ownership (TCO)

| Cost Component | API Model | Self-Hosted |
|---------------|-----------|-------------|
| Inference | $/token | GPU hours |
| Infrastructure | $0 | $2-50K/month |
| Engineering | Low | High (MLOps team) |
| Scaling | Automatic | Manual/K8s |
| Data privacy | Data leaves org | Stays internal |
| Vendor lock-in | High | None |

**Break-even**: Self-hosting wins at ~$10K+/month API spend.

### 1.5 Model Capabilities Matrix

| Model | Context | Function Calling | Vision | Code | Languages | Open Source |
|-------|---------|-----------------|--------|------|-----------|-------------|
| GPT-4o | 128K | ✅ | ✅ | ★★★★★ | 50+ | ❌ |
| Claude-3.5 | 200K | ✅ | ✅ | ★★★★★ | 30+ | ❌ |
| LLaMA-3.1-70B | 128K | ✅ | ❌ | ★★★★ | 8 | ✅ |
| Gemma-2-27B | 8K | ❌ | ❌ | ★★★ | 1 | ✅ |
| Mistral-Large | 128K | ✅ | ❌ | ★★★★ | 10+ | ❌ |
| Qwen-2.5-72B | 128K | ✅ | ❌ | ★★★★ | 29 | ✅ |

### 1.6 When to Fine-Tune vs Prompt-Engineer vs Use Bigger Model

```
Decision Tree:
├── Does the task need domain-specific knowledge?
│   ├── YES → Is the knowledge in documents? → RAG
│   └── YES → Is it behavioral/stylistic? → Fine-tune
├── Is the model failing with good prompts?
│   ├── YES → Try bigger model first (cheaper than fine-tuning)
│   └── NO → Prompt engineering is sufficient
└── Need consistent structured output?
    └── Function calling / constrained decoding
```

### 1.7 Scaling Laws

**Chinchilla (2022)**: Optimal training uses ~20 tokens per parameter.
- 7B model → 140B tokens
- 70B model → 1.4T tokens

**Practical implication**: A well-trained small model can beat a poorly-trained large one.

---

## Part 2: Implementation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Benchmark data for 8 models (from public benchmarks + pricing pages)
models_data = pd.DataFrame([
    {"model": "GPT-4o", "accuracy": 0.94, "latency_ms": 1200, "cost_input_per_1M": 2.50, "cost_output_per_1M": 10.0, "context_window": 128000, "function_calling": True, "open_source": False},
    {"model": "GPT-4o-mini", "accuracy": 0.88, "latency_ms": 600, "cost_input_per_1M": 0.15, "cost_output_per_1M": 0.60, "context_window": 128000, "function_calling": True, "open_source": False},
    {"model": "Claude-3.5-Sonnet", "accuracy": 0.93, "latency_ms": 1100, "cost_input_per_1M": 3.0, "cost_output_per_1M": 15.0, "context_window": 200000, "function_calling": True, "open_source": False},
    {"model": "Claude-3-Haiku", "accuracy": 0.85, "latency_ms": 400, "cost_input_per_1M": 0.25, "cost_output_per_1M": 1.25, "context_window": 200000, "function_calling": True, "open_source": False},
    {"model": "LLaMA-3.1-70B", "accuracy": 0.89, "latency_ms": 800, "cost_input_per_1M": 0.0, "cost_output_per_1M": 0.0, "context_window": 128000, "function_calling": True, "open_source": True},
    {"model": "LLaMA-3.1-8B", "accuracy": 0.79, "latency_ms": 200, "cost_input_per_1M": 0.0, "cost_output_per_1M": 0.0, "context_window": 128000, "function_calling": False, "open_source": True},
    {"model": "Gemma-2-27B", "accuracy": 0.83, "latency_ms": 500, "cost_input_per_1M": 0.0, "cost_output_per_1M": 0.0, "context_window": 8192, "function_calling": False, "open_source": True},
    {"model": "Mistral-Large", "accuracy": 0.90, "latency_ms": 900, "cost_input_per_1M": 2.0, "cost_output_per_1M": 6.0, "context_window": 128000, "function_calling": True, "open_source": False},
])
models_data

### Weighted Scoring Framework

In [ ]:
use_cases = {
    "enterprise_accuracy": {"accuracy": 0.5, "latency": 0.1, "cost": 0.2, "context": 0.2},
    "realtime_chatbot": {"accuracy": 0.2, "latency": 0.5, "cost": 0.2, "context": 0.1},
    "batch_processing": {"accuracy": 0.3, "latency": 0.05, "cost": 0.6, "context": 0.05},
    "long_document_qa": {"accuracy": 0.3, "latency": 0.1, "cost": 0.1, "context": 0.5},
    "budget_startup": {"accuracy": 0.25, "latency": 0.15, "cost": 0.5, "context": 0.1},
}

def normalize(s, invert=False):
    normed = (s - s.min()) / (s.max() - s.min() + 1e-9)
    return 1 - normed if invert else normed

def score_models(df, weights):
    scored = df[["model"]].copy()
    scored["score"] = (
        weights["accuracy"] * normalize(df["accuracy"]) +
        weights["latency"] * normalize(df["latency_ms"], invert=True) +
        weights["cost"] * normalize(df["cost_input_per_1M"], invert=True) +
        weights["context"] * normalize(df["context_window"])
    )
    return scored.sort_values("score", ascending=False)

for uc, w in use_cases.items():
    print(f"\n{'='*40}\n{uc.upper()}\n{'='*40}")
    print(score_models(models_data, w).to_string(index=False))

### Pareto Frontier

In [ ]:
def find_pareto(df, minimize_cols, maximize_cols):
    is_pareto = np.ones(len(df), dtype=bool)
    for i in range(len(df)):
        for j in range(len(df)):
            if i == j: continue
            dominates = True
            strictly_better = False
            for col in maximize_cols:
                if df.iloc[j][col] < df.iloc[i][col]: dominates = False
                if df.iloc[j][col] > df.iloc[i][col]: strictly_better = True
            for col in minimize_cols:
                if df.iloc[j][col] > df.iloc[i][col]: dominates = False
                if df.iloc[j][col] < df.iloc[i][col]: strictly_better = True
            if dominates and strictly_better:
                is_pareto[i] = False
                break
    return df[is_pareto]

pareto = find_pareto(models_data, ["cost_input_per_1M", "latency_ms"], ["accuracy"])
print(f"Pareto-optimal models ({len(pareto)}/{len(models_data)}):")
print(pareto[["model", "accuracy", "latency_ms", "cost_input_per_1M"]].to_string(index=False))

# Plot
fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(models_data["cost_input_per_1M"], models_data["accuracy"], s=100, c="lightblue", edgecolors="black", zorder=5)
ax.scatter(pareto["cost_input_per_1M"], pareto["accuracy"], s=150, c="red", edgecolors="black", zorder=6, label="Pareto Optimal")
for _, r in models_data.iterrows():
    ax.annotate(r["model"], (r["cost_input_per_1M"], r["accuracy"]), textcoords="offset points", xytext=(5,5), fontsize=8)
ax.set_xlabel("Cost ($/1M input tokens)")
ax.set_ylabel("Accuracy")
ax.set_title("Pareto Frontier: Cost vs Accuracy")
ax.legend()
plt.grid(True, alpha=0.3)
plt.show()

### Decision Matrix Heatmap

In [ ]:
score_matrix = pd.DataFrame({"model": models_data["model"]})
for uc, w in use_cases.items():
    scores = score_models(models_data, w)
    score_matrix = score_matrix.merge(scores.rename(columns={"score": uc}), on="model")
score_matrix = score_matrix.set_index("model")

plt.figure(figsize=(10, 6))
sns.heatmap(score_matrix, annot=True, fmt=".3f", cmap="YlGn", vmin=0, vmax=1)
plt.title("Model Suitability Score by Use Case")
plt.tight_layout()
plt.show()

### Recommendation Engine

In [ ]:
def recommend(requirements: dict) -> str:
    df = models_data.copy()
    if requirements.get("min_accuracy"): df = df[df["accuracy"] >= requirements["min_accuracy"]]
    if requirements.get("max_latency_ms"): df = df[df["latency_ms"] <= requirements["max_latency_ms"]]
    if requirements.get("max_cost"): df = df[df["cost_input_per_1M"] <= requirements["max_cost"]]
    if requirements.get("needs_function_calling"): df = df[df["function_calling"]]
    if requirements.get("min_context"): df = df[df["context_window"] >= requirements["min_context"]]
    if requirements.get("open_source_only"): df = df[df["open_source"]]
    if df.empty: return "No model meets all constraints. Relax requirements."
    best = df.sort_values("accuracy", ascending=False).iloc[0]
    return f"✅ Recommended: {best['model']} (accuracy={best['accuracy']}, latency={best['latency_ms']}ms, cost=${best['cost_input_per_1M']}/1M)"

print(recommend({"min_accuracy": 0.85, "needs_function_calling": True, "max_cost": 1.0}))
print(recommend({"max_latency_ms": 300, "open_source_only": True}))
print(recommend({"min_accuracy": 0.90, "min_context": 100000}))

## Part 3: Key Takeaways

- **No universal best model** — selection depends on constraints
- **Pareto frontier** narrows candidates objectively
- **Weighted scoring** makes trade-offs explicit and auditable
- **TCO analysis** often favors open-source at scale
- **Always benchmark on YOUR data** — public benchmarks don't reflect your domain

### Next → 03_vector_db.ipynb